In [1]:
"""
MODEL LOOKS TERRIBLE – FIND THE ENCODER BUG
 • 1 000 rows, 5 colours, 5-class label
 • Train split deliberately lacks ‘cyan’ and ‘magenta’
 • BUG: pd.factorize() called separately on train and test
   (order-of-appearance mapping)  → test accuracy ≈ 0.20
 • FIX: use the mapping learned on train for the test split
"""
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model   import LogisticRegression
from sklearn.metrics        import accuracy_score

rng      = np.random.RandomState(0)
classes  = np.array(['red','green','blue','cyan','magenta'])

# ------------------------------------------------------------
# 1.  make a data set whose target is strongly tied to colour
# ------------------------------------------------------------
n = 1_000
X = pd.DataFrame({"colour": rng.choice(classes, n)})

def noisy_identity(col):
    # correct label 80 % of the time, otherwise random wrong class
    y = col.copy()
    mask = rng.rand(n) > 0.80
    for i in np.where(mask)[0]:
        y.iat[i] = rng.choice(classes[classes != y.iat[i]])
    return y

y = noisy_identity(X['colour'])

# ------------------------------------------------------------
# 2.  train/test split WITHOUT shuffling
#     first 700 rows = train  (only red/green/blue)
#     last 300 rows  = test   (contains cyan & magenta)
# ------------------------------------------------------------
X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.30, shuffle=False)

# sanity-check the category sets
print("Train categories:", X_tr['colour'].unique())
print("Test  categories:", X_te['colour'].unique(), "\n")



Train categories: ['magenta' 'red' 'cyan' 'green' 'blue']
Test  categories: ['red' 'blue' 'green' 'magenta' 'cyan'] 



In [3]:
# ------------------------------------------------------------
# 3.  BUG – encode each split independently  (order matters!)
# ------------------------------------------------------------
def encode_series(s):
    codes, uniques = pd.factorize(s, sort=False)
    return codes, uniques

X_tr_enc, uniq_tr = encode_series(X_tr['colour'])      # fit here
X_te_enc, _       = encode_series(X_te['colour'])      # ❌ re-fit here

clf = LogisticRegression(max_iter=400,
                         multi_class='multinomial').fit(
                         X_tr_enc.reshape(-1,1), y_tr)

print("Accuracy WITH bug :", round(
        accuracy_score(y_te, clf.predict(X_te_enc.reshape(-1,1))), 3))
print(uniq_tr, "\n")
print(_)


Accuracy WITH bug : 0.05
Index(['magenta', 'red', 'cyan', 'green', 'blue'], dtype='object') 

Index(['red', 'blue', 'green', 'magenta', 'cyan'], dtype='object')


/home/edward99/github/datenvorbearbeitung/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [ ]:


# ------------------------------------------------------------
# 4.  FIX – transform test split with the train mapping only
# ------------------------------------------------------------
mapping = {c: i for i, c in enumerate(uniq_tr)}
X_te_fixed = X_te['colour'].map(mapping).fillna(-1).astype(int)

print("Accuracy after fix:", round(
        accuracy_score(y_te, clf.predict(X_te_fixed.values.reshape(-1,1))), 3))

# Model too bad Example 2

**Bug:** Features and target are shuffled independently, breaking their correspondence.

**Result:** Model accuracy ≈ random guessing (0.20)

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Create a simple dataset where feature strongly predicts target
rng = np.random.RandomState(42)
n = 2000

# Feature: age groups
X = pd.DataFrame({
    'age': rng.randint(18, 80, n),
    'score': rng.rand(n) * 100
})

# Target: age category (strongly correlated with age)
def get_category(age):
    if age < 30: return 'young'
    elif age < 45: return 'adult'
    elif age < 60: return 'middle'
    else: return 'senior'

y = X['age'].apply(get_category)        # this defines the field we want to predict

print("Original correlation check:")
print(pd.crosstab(X['age'] // 15, y))

In [ ]:
# ❌ BUG: Shuffle features and target independently!
X_shuffled = X.sample(frac=1, random_state=98).reset_index(drop=True)
y_shuffled = y.sample(frac=1, random_state=77).reset_index(drop=True)  # Different seed! -> this is where we shuffle the field we want to predict, 
#if use different random state, then the label mapping will be wrong all over the dataset
"""
# Result after combining:
    features_from  label_from   WRONG!
0   Carol (65)     Bob (adult)  ← 65yo labeled as "adult"
1   Alice (25)     Eve (middle) ← 25yo labeled as "middle"
2   Eve (50)       Carol (senior) ← 50yo labeled as "senior"
3   David (30)     Alice (young) ← 30yo labeled as "young" (lucky!)
4   Bob (45)       David (young) ← 45yo labeled as "young"
"""



In [ ]:
# Now features don't match their labels
X_train, X_test, y_train, y_test = train_test_split(
    X_shuffled, y_shuffled, test_size=0.25, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("\n❌ WITH BUG (shuffled independently):")
print(f"Train accuracy: {model.score(X_train, y_train):.3f}")
print(f"Test accuracy: {model.score(X_test, y_test):.3f}")
print(f"Random baseline: {1/len(y.unique()):.3f}")

In [ ]:
# ✓ FIX: Don't shuffle independently, or shuffle together
# Method 1: Don't shuffle at all if indices already align
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Method 2: If you must shuffle, shuffle together
# combined = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)
# combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

model_fixed = RandomForestClassifier(n_estimators=100, random_state=42)
model_fixed.fit(X_train, y_train)

print("\n✓ AFTER FIX:")
print(f"Train accuracy: {model_fixed.score(X_train, y_train):.3f}")
print(f"Test accuracy: {model_fixed.score(X_test, y_test):.3f}")